# 🛒 Intelligent Shopping & Market Basket Recommendation System

## Notebook 02 — Data Understanding

**Dataset:** Instacart Online Grocery Basket Dataset

### Objective

The objective of this notebook is to understand the structure,
quality, relationships, and basic characteristics of the Instacart
grocery dataset before performing exploratory data analysis,
market basket analysis, and recommendation modeling.

# Step 1 — Import Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully!")

Libraries imported successfully!


# Step 2 — Dataset Paths

The dataset contains multiple relational tables including products,
orders, aisles, departments, and order-product transactions.

In [3]:
# Dataset paths

DATA_PATH = Path("../data/raw")

aisles_path = DATA_PATH / "aisles.csv"
departments_path = DATA_PATH / "departments.csv"
products_path = DATA_PATH / "products.csv"
orders_path = DATA_PATH / "orders.csv"
prior_path = DATA_PATH / "order_products__prior.csv"
train_path = DATA_PATH / "order_products__train.csv"

print("Dataset paths defined successfully!")

Dataset paths defined successfully!


# Step 3 — Load Manageable Datasets

In [4]:
# Load manageable datasets

aisles = pd.read_csv(aisles_path)
departments = pd.read_csv(departments_path)
products = pd.read_csv(products_path)
orders = pd.read_csv(orders_path)

# Train transaction data is much smaller than prior
order_products_train = pd.read_csv(train_path)

print("Manageable datasets loaded successfully!")

Manageable datasets loaded successfully!


# Step 4 — Dataset Dimensions

The dimensions of each dataset are examined to understand
the scale of the available data.

In [5]:
print("Aisles shape:", aisles.shape)
print("Departments shape:", departments.shape)
print("Products shape:", products.shape)
print("Orders shape:", orders.shape)
print("Order Products Train shape:", order_products_train.shape)

# Count rows in the huge prior dataset without loading it
prior_rows = sum(
    1 for _ in open(prior_path, encoding="utf-8")
) - 1

print("Order Products Prior rows:", prior_rows)

Aisles shape: (134, 2)
Departments shape: (21, 2)
Products shape: (49688, 4)
Orders shape: (3421083, 7)
Order Products Train shape: (1384617, 4)
Order Products Prior rows: 32434489


# Step 5 — Inspect First Records

The first few records of each table are inspected to understand
the structure and meaning of the variables.

In [6]:
print("AISLES")
display(aisles.head())

print("\nDEPARTMENTS")
display(departments.head())

print("\nPRODUCTS")
display(products.head())

print("\nORDERS")
display(orders.head())

print("\nORDER PRODUCTS TRAIN")
display(order_products_train.head())

AISLES


,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation



DEPARTMENTS


,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol



PRODUCTS


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13



ORDERS


,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0



ORDER PRODUCTS TRAIN


,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1


# Step 6 — Columns and Data Types

Column names and data types are inspected to identify
the variables available for further analysis.

In [7]:
datasets = {
    "aisles": aisles,
    "departments": departments,
    "products": products,
    "orders": orders,
    "order_products_train": order_products_train
}

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("Columns:", df.columns.tolist())
    print("Data types:")
    print(df.dtypes)


AISLES
Columns: ['aisle_id', 'aisle']
Data types:
aisle_id     int64
aisle       object
dtype: object

DEPARTMENTS
Columns: ['department_id', 'department']
Data types:
department_id     int64
department       object
dtype: object

PRODUCTS
Columns: ['product_id', 'product_name', 'aisle_id', 'department_id']
Data types:
product_id        int64
product_name     object
aisle_id          int64
department_id     int64
dtype: object

ORDERS
Columns: ['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']
Data types:
order_id                    int64
user_id                     int64
eval_set                   object
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
dtype: object

ORDER_PRODUCTS_TRAIN
Columns: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']
Data types:
order_id             int64
product_id           int64
add_to_cart_order

# Step 7 — Missing Value Analysis

Missing values are checked across all manageable datasets.
The large prior transaction table will be checked using
chunk processing to avoid excessive memory usage.

In [8]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.isnull().sum())


AISLES
aisle_id    0
aisle       0
dtype: int64

DEPARTMENTS
department_id    0
department       0
dtype: int64

PRODUCTS
product_id       0
product_name     0
aisle_id         0
department_id    0
dtype: int64

ORDERS
order_id                       0
user_id                        0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64

ORDER_PRODUCTS_TRAIN
order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64


In [9]:
# Missing values in large prior dataset using chunks

prior_missing = {}

for chunk in pd.read_csv(prior_path, chunksize=500_000):

    for column in chunk.columns:
        prior_missing[column] = (
            prior_missing.get(column, 0)
            + chunk[column].isnull().sum()
        )

print("ORDER PRODUCTS PRIOR")
print(prior_missing)

ORDER PRODUCTS PRIOR
{'order_id': np.int64(0), 'product_id': np.int64(0), 'add_to_cart_order': np.int64(0), 'reordered': np.int64(0)}


# Step 8 — Duplicate Record Analysis

Duplicate records are checked in manageable datasets.
The very large prior transaction table is handled separately
using its natural transaction keys.

In [10]:
# Duplicate check for manageable datasets

for name, df in datasets.items():
    print(
        f"{name}: {df.duplicated().sum()} duplicate rows"
    )

aisles: 0 duplicate rows
departments: 0 duplicate rows
products: 0 duplicate rows
orders: 0 duplicate rows
order_products_train: 0 duplicate rows


### Instead

In [11]:
# Check duplicate order-product pairs using chunk processing
# without storing millions of pairs in a Python set.

prior_path = "../data/raw/order_products__prior.csv"

duplicate_pairs = 0

for chunk in pd.read_csv(
    prior_path,
    usecols=["order_id", "product_id"],
    chunksize=500_000,
    dtype={
        "order_id": "int32",
        "product_id": "int32"
    }
):
    # Count duplicate rows within the current chunk
    duplicate_pairs += chunk.duplicated(
        subset=["order_id", "product_id"]
    ).sum()

print(
    "Duplicate order-product pairs found within chunks:",
    duplicate_pairs
)

Duplicate order-product pairs found within chunks: 0


In [21]:
datasets = {
    "Aisles": aisles,
    "Departments": departments,
    "Products": products,
    "Orders": orders
}

for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    print(f"{name}: {duplicate_count:,} duplicate rows")

Aisles: 0 duplicate rows
Departments: 0 duplicate rows
Products: 0 duplicate rows
Orders: 0 duplicate rows


# Step 9 — Table Relationship Verification

The relationships between the relational tables are verified
using primary and foreign key columns.

### Product IDs

In [22]:
print(
    "Unique product IDs in products:",
    products["product_id"].nunique()
)

print(
    "Unique product IDs in train:",
    order_products_train["product_id"].nunique()
)

Unique product IDs in products: 49688
Unique product IDs in train: 39123


### Order IDs

In [23]:
print(
    "Unique order IDs in orders:",
    orders["order_id"].nunique()
)

print(
    "Unique order IDs in train:",
    order_products_train["order_id"].nunique()
)

Unique order IDs in orders: 3421083
Unique order IDs in train: 131209


### Product ID validity

In [24]:
product_id_check = order_products_train["product_id"].isin(
    products["product_id"]
).mean()

print(
    "Percentage of train product IDs present in products:",
    round(product_id_check * 100, 2),
    "%"
)

Percentage of train product IDs present in products: 100.0 %


### Order ID Validity

In [25]:
order_id_check = order_products_train["order_id"].isin(
    orders["order_id"]
).mean()

print(
    "Percentage of train order IDs present in orders:",
    round(order_id_check * 100, 2),
    "%"
)

Percentage of train order IDs present in orders: 100.0 %


# Step 10 — Product → Department → Aisle Relationship

In [26]:
# Merge products with department and aisle information

product_structure = products.merge(
    departments,
    on="department_id",
    how="left"
).merge(
    aisles,
    on="aisle_id",
    how="left"
)

display(product_structure.head())

,product_id,product_name,aisle_id,department_id,department,aisle
0,1,Chocolate Sandwich Cookies,61,19,snacks,cookies cakes
1,2,All-Seasons Salt,104,13,pantry,spices seasonings
2,3,Robust Golden Unsweetened Oolong Tea,94,7,beverages,tea
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,frozen,frozen meals
4,5,Green Chile Anytime Sauce,5,13,pantry,marinades meat preparation


### Check missing relationships:

In [27]:
print(
    "Missing department names:",
    product_structure["department"].isnull().sum()
)

print(
    "Missing aisle names:",
    product_structure["aisle"].isnull().sum()
)

Missing department names: 0
Missing aisle names: 0


# Step 11 — Basic Datasets Statistics

Basic descriptive statistics are calculated for the manageable
datasets. Large transaction data will be analyzed using
chunk processing.

In [28]:
print("PRODUCTS STATISTICS")
display(products.describe())

print("\nORDERS STATISTICS")
display(orders.describe())

print("\nTRAIN ORDER-PRODUCT STATISTICS")
display(order_products_train.describe())

PRODUCTS STATISTICS


,product_id,aisle_id,department_id
count,49688.000000,49688.000000,49688.000000
mean,24844.500000,67.769582,11.728687
std,14343.834425,38.316162,5.850410
min,1.000000,1.000000,1.000000
25%,12422.750000,35.000000,7.000000
50%,24844.500000,69.000000,13.000000
75%,37266.250000,100.000000,17.000000
max,49688.000000,134.000000,21.000000



ORDERS STATISTICS


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
count,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.214874e+06
mean,1.710542e+06,1.029782e+05,1.715486e+01,2.776219e+00,1.345202e+01,1.111484e+01
std,9.875817e+05,5.953372e+04,1.773316e+01,2.046829e+00,4.226088e+00,9.206737e+00
min,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,8.552715e+05,5.139400e+04,5.000000e+00,1.000000e+00,1.000000e+01,4.000000e+00
50%,1.710542e+06,1.026890e+05,1.100000e+01,3.000000e+00,1.300000e+01,7.000000e+00
75%,2.565812e+06,1.543850e+05,2.300000e+01,5.000000e+00,1.600000e+01,1.500000e+01
max,3.421083e+06,2.062090e+05,1.000000e+02,6.000000e+00,2.300000e+01,3.000000e+01



TRAIN ORDER-PRODUCT STATISTICS


,order_id,product_id,add_to_cart_order,reordered
count,1.384617e+06,1.384617e+06,1.384617e+06,1.384617e+06
mean,1.706298e+06,2.555624e+04,8.758044e+00,5.985944e-01
std,9.897326e+05,1.412127e+04,7.423936e+00,4.901829e-01
min,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00
25%,8.433700e+05,1.338000e+04,3.000000e+00,0.000000e+00
50%,1.701880e+06,2.529800e+04,7.000000e+00,1.000000e+00
75%,2.568023e+06,3.794000e+04,1.200000e+01,1.000000e+00
max,3.421070e+06,4.968800e+04,8.000000e+01,1.000000e+00


In [29]:
# print("PRODUCTS STATISTICS")
display(products.describe())

print("\nORDERS STATISTICS")
display(orders.describe())

print("\nTRAIN ORDER-PRODUCT STATISTICS")
display(order_products_train.describe())

,product_id,aisle_id,department_id
count,49688.000000,49688.000000,49688.000000
mean,24844.500000,67.769582,11.728687
std,14343.834425,38.316162,5.850410
min,1.000000,1.000000,1.000000
25%,12422.750000,35.000000,7.000000
50%,24844.500000,69.000000,13.000000
75%,37266.250000,100.000000,17.000000
max,49688.000000,134.000000,21.000000



ORDERS STATISTICS


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
count,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.214874e+06
mean,1.710542e+06,1.029782e+05,1.715486e+01,2.776219e+00,1.345202e+01,1.111484e+01
std,9.875817e+05,5.953372e+04,1.773316e+01,2.046829e+00,4.226088e+00,9.206737e+00
min,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,8.552715e+05,5.139400e+04,5.000000e+00,1.000000e+00,1.000000e+01,4.000000e+00
50%,1.710542e+06,1.026890e+05,1.100000e+01,3.000000e+00,1.300000e+01,7.000000e+00
75%,2.565812e+06,1.543850e+05,2.300000e+01,5.000000e+00,1.600000e+01,1.500000e+01
max,3.421083e+06,2.062090e+05,1.000000e+02,6.000000e+00,2.300000e+01,3.000000e+01



TRAIN ORDER-PRODUCT STATISTICS


,order_id,product_id,add_to_cart_order,reordered
count,1.384617e+06,1.384617e+06,1.384617e+06,1.384617e+06
mean,1.706298e+06,2.555624e+04,8.758044e+00,5.985944e-01
std,9.897326e+05,1.412127e+04,7.423936e+00,4.901829e-01
min,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00
25%,8.433700e+05,1.338000e+04,3.000000e+00,0.000000e+00
50%,1.701880e+06,2.529800e+04,7.000000e+00,1.000000e+00
75%,2.568023e+06,3.794000e+04,1.200000e+01,1.000000e+00
max,3.421070e+06,4.968800e+04,8.000000e+01,1.000000e+00


# Step 12 — Large Prior Dataset Statistics

In [30]:
total_records = 0
total_reordered = 0
total_add_to_cart = 0

min_add_to_cart = float("inf")
max_add_to_cart = float("-inf")

for chunk in pd.read_csv(
    prior_path,
    chunksize=500_000
):

    total_records += len(chunk)

    total_reordered += chunk["reordered"].sum()

    total_add_to_cart += (
        chunk["add_to_cart_order"].sum()
    )

    min_add_to_cart = min(
        min_add_to_cart,
        chunk["add_to_cart_order"].min()
    )

    max_add_to_cart = max(
        max_add_to_cart,
        chunk["add_to_cart_order"].max()
    )

print("Total records:", total_records)

print(
    "Reordered products:",
    total_reordered
)

print(
    "Reorder percentage:",
    round(
        total_reordered / total_records * 100,
        2
    ),
    "%"
)

print(
    "Minimum add-to-cart order:",
    min_add_to_cart
)

print(
    "Maximum add-to-cart order:",
    max_add_to_cart
)

Total records: 32434489
Reordered products: 19126536
Reorder percentage: 58.97 %
Minimum add-to-cart order: 1
Maximum add-to-cart order: 145


# Step 13 — Initial Observations

## Key Observations

1. The dataset contains multiple relational tables covering
   products, aisles, departments, orders, and order-product
   transactions.

2. The order-product prior dataset is significantly larger
   than the other tables and therefore requires chunk-based
   processing.

3. Product IDs in the transaction data are linked to the
   products table.

4. Order IDs in the transaction data are linked to the
   orders table.

5. Product records can be connected with department and aisle
   information.

6. Missing-value and duplicate checks are performed before
   further analysis.

7. Reorder information provides an important signal for
   understanding repeat purchasing behavior.

8. The relational structure of the dataset is suitable for
   market basket analysis and recommendation modeling.